# Lab 05: word count with Spark

Run the cells from top to bottom (Shift+Enter). Every cell works as it is.

Spark runs in **local mode** inside this notebook container. It still reads and writes the HDFS from the earlier labs.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("lab-05-spark-and-jupyter")
    .config("spark.hadoop.dfs.replication", "1")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
print("Spark", spark.version)
print("Spark UI:", "http://spark.localhost")

## 1. Make some input in HDFS

20,000 short lines of random words from a small vocabulary. Spark writes them to HDFS as text files, so this lab does not depend on any other lab.

In [ ]:
import random

BASE = "hdfs://namenode:8020/user/lab/05"

vocab = ["data", "hadoop", "hdfs", "yarn", "spark", "hive", "block", "replica", "namenode", "datanode",
         "parquet", "iceberg", "kafka", "cluster", "shuffle", "stage"]
weights = [1 / (rank + 1) for rank in range(len(vocab))]

random.seed(42)
lines = [
    " ".join(random.choices(vocab, weights=weights, k=random.randint(6, 12)))
    for _ in range(20_000)
]

(spark.createDataFrame([(line,) for line in lines], "value string")
      .repartition(2)
      .write.mode("overwrite").text(f"{BASE}/input"))
print("wrote", len(lines), "lines to", f"{BASE}/input")

## 2. Read it back and count words

`explode` turns each line into one row per word. `groupBy("word").count()` is the same idea as the map and reduce steps in lab 02.

In [ ]:
text = spark.read.text(f"{BASE}/input")
text.show(3, truncate=False)

words = text.select(F.explode(F.split(F.lower("value"), r"\s+")).alias("word")).filter("word <> ''")
counts = words.groupBy("word").count().orderBy(F.desc("count"))
counts.show(10)

## 3. Read the plan

`explain()` prints what Spark will do. Look for `Exchange` (the shuffle), `HashAggregate` (the partial and final count) and `FileScan` (reading HDFS).

In [ ]:
counts.explain()

Now open http://spark.localhost, then **Jobs** and **SQL / DataFrame**. The last query is listed there with its stages and the shuffle between them.

## 4. Chart it

`toPandas()` pulls the (small) result into this notebook. Never call it on a big table.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top = counts.limit(10).toPandas()
ax = sns.barplot(data=top, y="word", x="count", color="#e8590c")
ax.set_title("Top words in /user/lab/05/input")
plt.tight_layout()
plt.show()

## 5. Write the results to HDFS

The full counts go out as Parquet (a columnar format, more in lab 06). A top-10 list goes out as one small CSV so you can read it with `hdfs dfs -cat`.

In [ ]:
counts.write.mode("overwrite").parquet(f"{BASE}/word-counts-parquet")

(counts.limit(10).coalesce(1)
       .write.mode("overwrite").option("header", True).csv(f"{BASE}/top-words"))

back = spark.read.parquet(f"{BASE}/word-counts-parquet")
print("rows in the Parquet copy:", back.count())
back.show(5)

In [ ]:
spark.stop()